In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
from datasets import load_dataset, DatasetDict
from transformers import pipeline, AutoTokenizer, AutoModelForMaskedLM, AutoModelForSequenceClassification
from peft import IA3Config, get_peft_model

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("mteb/imdb")

# ds = DatasetDict({
#     "train": ds["train"].shuffle(seed=42).select(range(500)),
#     "test": ds["test"].shuffle(seed=42).select(range(200)),
# })

In [3]:
def setup_config(target_modules, feedforward_modules=None):
    config = IA3Config(
        target_modules = target_modules,
        task_type="SEQ_CLS",
        feedforward_modules=feedforward_modules if feedforward_modules is not None else [],
    )
    return config
    


In [4]:
from transformers import TrainingArguments, Trainer
import time
import numpy as np
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
import os
from transformers import BitsAndBytesConfig
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_target_modules(model_id):
    model_id = model_id.lower()
    if "roberta" in model_id:
        return ["query", "value"]
    elif "deberta" in model_id:
        return ["query_proj", "value_proj"]
    elif "distilbert" in model_id:
        return ["q_lin", "v_lin"]
    else:
        # Default fallback for most BERT-like models
        return ["query", "value"]


def prepare_datasets(model_name, ds):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)
    
    tokenized_dataset = ds.map(tokenize_function, batched=True)
    
    cols_to_remove = [col for col in tokenized_dataset["train"].column_names if col not in ["input_ids", "attention_mask", "label"]]
    tokenized_dataset = tokenized_dataset.remove_columns(cols_to_remove)
    tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
    tokenized_dataset.set_format("torch")
    
    train_valid = tokenized_dataset["train"].train_test_split(test_size=0.1, seed=42)
    
    train_ds = train_valid["train"]
    val_ds = train_valid["test"]
    test_ds = tokenized_dataset["test"] 

    return train_ds, val_ds, test_ds



In [7]:
logs = []
def training_pipeline_adapter(model_id, config, run_name,seed, train_ds, val_ds, test_ds):
    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=2,
        use_safetensors=True
    )
    set_seed(seed)

    print(f"--- Training Run: {run_name} ---")
    model = get_peft_model(base_model, config)
    model.print_trainable_parameters()
    
    batch_size = 16
    learning_rate = 1e-5

    safe_model_name = model_id.split("/")[-1]
    out_dir = f"./results/{safe_model_name}_{run_name}_seed{seed}"
    args = TrainingArguments(
        output_dir=out_dir,
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=4,
        per_device_eval_batch_size=batch_size,
        bf16=True,
        num_train_epochs=2,
        logging_steps=10,
        load_best_model_at_end=True,
        label_names=["labels"],
    )

    trainer = Trainer(
        model,
        args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
    )
    
    os.system("nvidia-smi")
    start = time.time()
    trainer.train()
    os.system("nvidia-smi")
    end = time.time()
    train_time = end - start
    if torch.cuda.is_available():
        peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 3)
    else:
        peak_vram = 0
    predictions = trainer.predict(test_ds)

    preds = np.argmax(predictions.predictions, axis=1)
    
    labels = predictions.label_ids
    
    acc = accuracy_score(labels, preds)
        
    f1 = f1_score(labels, preds, average="macro")

    print(f"[{run_name}] Accuracy: {acc:.4f}")
    print(f"[{run_name}] F1 Score: {f1:.4f}")
    print(f"[{run_name}] Training Time: {train_time:.2f} seconds\n")

    logs.append({
        "seed": seed,
        "lr": learning_rate,
        "run_name": run_name,
        "target": str(config.target_modules),
        "ffn": str(config.feedforward_modules),
        "accuracy": acc,
        "f1": f1,
        "train_time": train_time,
        "vram": peak_vram
    })


    del model, trainer, base_model
    torch.cuda.empty_cache()


In [8]:

model_ids = ["FacebookAI/roberta-base", "microsoft/deberta-v3-base", "distilbert/distilbert-base-uncased"]


for model_id in model_ids:
    configs_to_test = {
        f"IA3_{model_id}": setup_config(get_target_modules(model_id)),
    }
    train_ds, val_ds, test_ds = prepare_datasets(model_id, ds)
    for run_name, config in configs_to_test.items():
        for seed in [42, 43]:
            training_pipeline_adapter(model_id, config, run_name, seed, train_ds, val_ds, test_ds)



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


--- Training Run: IA3_FacebookAI/roberta-base ---
trainable params: 610,562 || all params: 125,257,732 || trainable%: 0.4874


Epoch,Training Loss,Validation Loss
1,2.745068,0.684701
2,2.714441,0.681830


[IA3_FacebookAI/roberta-base] Accuracy: 0.7492
[IA3_FacebookAI/roberta-base] F1 Score: 0.7428
[IA3_FacebookAI/roberta-base] Training Time: 351.92 seconds



...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


--- Training Run: IA3_distilbert/distilbert-base-uncased ---
trainable params: 601,346 || all params: 67,556,356 || trainable%: 0.8901


Epoch,Training Loss,Validation Loss
1,2.577649,0.638020
2,2.479687,0.616741


[IA3_distilbert/distilbert-base-uncased] Accuracy: 0.7856
[IA3_distilbert/distilbert-base-uncased] F1 Score: 0.7856
[IA3_distilbert/distilbert-base-uncased] Training Time: 187.38 seconds



In [10]:
experiments = [
    {
        "id": "EXP01",
        "model_id": "FacebookAI/roberta-base",
        "attention": ["value"],
        "feedforward": ["intermediate.dense", "output.dense"],
        "name": "roberta_ffn"
    },
    {
        "id": "EXP02",
        "model_id": "microsoft/deberta-v3-base",
        "attention": ["value_proj"],
        "feedforward": ["intermediate.dense", "output.dense"],
        "name": "deberta_ffn"
    },
    {
        "id": "EXP03",
        "model_id": "distilbert/distilbert-base-uncased",
        "attention": ["v_lin"],
        "feedforward": ["ffn.lin1", "ffn.lin2"],
        "name": "distil_ffn"
    }
]



for exp in experiments:
    model_id = exp['model_id']
    run_name = exp['name']
    all_targets = exp['attention'] + exp['feedforward']
    config = setup_config(all_targets, exp['feedforward'])
    train_ds, val_ds, test_ds = prepare_datasets(model_id, ds)
    for seed in [42, 43]:
        training_pipeline_adapter(model_id, config, run_name, seed, train_ds, val_ds, test_ds)




Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


--- Training Run: roberta_ffn ---
trainable params: 656,642 || all params: 125,303,812 || trainable%: 0.5240


Epoch,Training Loss,Validation Loss
1,2.744775,0.684621
2,2.713696,0.681699


[roberta_ffn] Accuracy: 0.7495
[roberta_ffn] F1 Score: 0.7430
[roberta_ffn] Training Time: 436.25 seconds



...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


--- Training Run: distil_ffn ---
trainable params: 619,778 || all params: 67,574,788 || trainable%: 0.9172


Epoch,Training Loss,Validation Loss
1,2.571423,0.636295
2,2.459204,0.611446


[distil_ffn] Accuracy: 0.7899
[distil_ffn] F1 Score: 0.7899
[distil_ffn] Training Time: 221.86 seconds



In [11]:
df = pd.DataFrame(logs)
df.to_csv("ia3_adapter_experiment_results.csv", index=False)